# SenseVoice remote server

Hosts the exact same SenseVoice (funasr) model `workers/verbal_worker.py`'s `_transcribe_alt` calls locally, but as an HTTP endpoint you can call from your own machine instead of loading it there. See `wikis/Verbal-Worker.md` for why (SenseVoice's torch-backed load is heavy — the whole point of this notebook is to keep it off your laptop).

**Before running:**
1. Runtime -> Change runtime type -> pick a GPU (T4 is fine, free tier) if you want faster inference. CPU works too, just slower — SenseVoice is genuinely light enough to run on CPU if no GPU is available.
2. Sign up for a free ngrok account at https://dashboard.ngrok.com/signup and grab your authtoken from https://dashboard.ngrok.com/get-started/your-authtoken — paste it into the `NGROK_AUTHTOKEN` cell below.
3. Pick your own `API_KEY` below (any random string) and copy the *same* value into your local `.env` as `SENSEVOICE_API_KEY` — this is the only thing stopping a random person who guesses your ngrok URL from using your compute.
4. **Optional, recommended**: skip re-downloading SenseVoice (~900MB) by reusing your local cache instead — see the "Restore cached model files from Drive" cell below. If you skip it, the model-loading cell just downloads fresh, which is slower but works fine on its own.

**Session lifetime:** Colab's free tier disconnects after a period of inactivity and has a hard runtime cap (~12h). This is meant for "I'm actively running an analysis batch right now" use, not a permanently-on service — re-run this notebook (you'll get a new ngrok URL each time) whenever you want to use it again, and update `SENSEVOICE_REMOTE_URL` locally to match.

**Chunked requests:** the client (`workers/verbal_worker.py`) sends audio in ~60s chunks, several per request, rather than one whole-file request — `/transcribe` below expects a *list* of files (`files=`), not a single `file=`. This is what lets FunASR's own `batch_size_s` batching actually do something, keeps each request's duration bounded regardless of video length, and gives you a log line per batch (both here in the Colab output and in the local `[verbal]` logs) instead of one long silent wait.

In [ ]:
!pip install -q funasr fastapi "uvicorn[standard]" pyngrok python-multipart

## Restore cached model files from Drive (optional)

Skip this cell entirely if you'd rather just let the next cell download SenseVoice fresh.

To use it: on your own machine, the model files are already cached (funasr/modelscope downloaded them there the first time SenseVoice ran locally) at `~/.cache/modelscope/models/`. Package them up and upload to Google Drive once:
```bash
cd ~
tar -czf sensevoice_models.tar.gz -C ~/.cache/modelscope models/
```
Then upload `sensevoice_models.tar.gz` to the root of your Google Drive (drag-drop at https://drive.google.com is fine — Drive handles a file this size far more reliably than Colab's direct browser upload widget would). Adjust the path below if you put it somewhere other than Drive's root.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.environ['MODELSCOPE_CACHE'] = '/root/.cache/modelscope'  # must match where AutoModel looks, below

ARCHIVE_PATH = '/content/drive/MyDrive/sensevoice_models.tar.gz'  # adjust if uploaded elsewhere

!mkdir -p /root/.cache/modelscope
!tar -xzf "{ARCHIVE_PATH}" -C /root/.cache/modelscope
print("Extracted. Contents:")
!ls -la /root/.cache/modelscope/models/

In [ ]:
# --- Configuration — edit these two before running ---
NGROK_AUTHTOKEN = "PASTE_YOUR_NGROK_AUTHTOKEN_HERE"
API_KEY = "PASTE_A_RANDOM_SECRET_HERE"  # must match SENSEVOICE_API_KEY in your local .env

assert NGROK_AUTHTOKEN != "PASTE_YOUR_NGROK_AUTHTOKEN_HERE", "Set NGROK_AUTHTOKEN above first"
assert API_KEY != "PASTE_A_RANDOM_SECRET_HERE", "Set API_KEY above first — pick any random string"

In [ ]:
import torch

from funasr import AutoModel

_SENSEVOICE_MODEL = "iic/SenseVoiceSmall"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading SenseVoice on {device}...")

# Identical config to VerbalWorker._get_sensevoice() locally — keep these in
# sync if that ever changes, so remote and local produce the same output.
# If the cache-restore cell above ran, AutoModel finds everything already
# in place and doesn't re-download; otherwise it downloads fresh here.
model = AutoModel(
    model=_SENSEVOICE_MODEL,
    trust_remote_code=True,
    vad_model="fsmn-vad",
    vad_kwargs={"max_single_segment_time": 30000},
    device=device,
    disable_update=True,
)
print("SenseVoice loaded and warm.")

In [ ]:
import shutil
import tempfile
import time
from pathlib import Path

from fastapi import FastAPI, File, Form, Header, HTTPException, UploadFile

app = FastAPI()


@app.get("/health")
def health():
    return {"status": "ok", "device": device}


@app.post("/transcribe")
async def transcribe(
    files: list[UploadFile] = File(...),
    lang_code: str = Form(...),
    x_api_key: str = Header(None),
):
    """
    Accepts a *list* of audio chunks in one request (see
    workers/verbal_worker.py's _transcribe_alt_remote docstring for why --
    short version: passing a list to model.generate() below is what makes
    FunASR actually batch them together on the GPU, instead of processing
    one whole file at a time). Returns one result per chunk, in the same
    order they were sent, so the client can offset each chunk's timestamps
    back onto the full audio's timeline.
    """
    if x_api_key != API_KEY:
        raise HTTPException(401, "bad or missing X-API-Key header")

    with tempfile.TemporaryDirectory() as tmp_dir:
        tmp_paths = []
        for i, f in enumerate(files):
            tmp_path = Path(tmp_dir) / (f.filename or f"chunk_{i}.wav")
            with open(tmp_path, "wb") as out:
                shutil.copyfileobj(f.file, out)
            tmp_paths.append(str(tmp_path))

        print(f"[sensevoice-server] Received {len(tmp_paths)} chunks -- starting generate()...")
        t0 = time.time()

        # Same call as VerbalWorker._transcribe_alt locally, except input is
        # a list here -- that's the whole point, see the docstring above.
        results = model.generate(
            input=tmp_paths,
            cache={},
            language=lang_code,
            use_itn=True,
            batch_size_s=300,
            output_timestamp=True,
        )
        print(f"[sensevoice-server] generate() done in {time.time() - t0:.1f}s for {len(tmp_paths)} chunks")

    # results is expected in the same order as tmp_paths (FunASR preserves
    # input order for batched/list input) -- if that ever stops being true,
    # the client-side length check in _send_chunk_batch will at least catch
    # a count mismatch, though not a silent reorder.
    chunks_out = []
    for item in results:
        words = item.get("words") or []
        timestamps = item.get("timestamp") or []
        tokens = []
        for word, (start_ms, end_ms) in zip(words, timestamps):
            tokens.append({
                "word": word,
                "start_s": start_ms / 1000.0,
                "end_s": end_ms / 1000.0,
                "confidence": 1.0,
            })
        chunks_out.append({"tokens": tokens})

    return {"chunks": chunks_out}


In [ ]:
import threading
import time

import uvicorn
from pyngrok import conf, ngrok

conf.get_default().auth_token = NGROK_AUTHTOKEN
public_url = ngrok.connect(8000, "http")
print(f"\n  Public URL: {public_url}\n")
print(f"  Set locally: SENSEVOICE_REMOTE_URL={public_url}/transcribe\n")

# Run uvicorn in a background thread rather than uvicorn.run() directly.
# Colab's notebook kernel already has its own asyncio event loop running on
# the main thread — uvicorn.run() calls asyncio.run() internally, and
# calling that on top of an already-running loop raises "asyncio.run()
# cannot be called from a running event loop". (nest_asyncio used to be the
# usual workaround, but it doesn't reliably patch newer uvicorn/Python
# 3.11+'s asyncio.Runner internals — this sidesteps the problem instead of
# patching around it: a new thread has no event loop of its own, so
# uvicorn's asyncio.run() inside it never conflicts with Colab's.)
config = uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="info")
server = uvicorn.Server(config)
thread = threading.Thread(target=server.run, daemon=True)
thread.start()
time.sleep(2)  # give it a moment to actually start listening

print("Server running in the background thread — this cell finishing is")
print("expected, the server keeps running for as long as this Colab")
print("runtime stays connected. Run the next cell to confirm it's up.")

In [ ]:
# Sanity check — confirms the server is actually listening and the ngrok
# tunnel is actually forwarding to it, end to end, before you go copy the
# URL into your local .env.
import requests

resp = requests.get(f"{public_url.public_url}/health", timeout=10)
resp.raise_for_status()
print(resp.json())